# Projeto de Mineração de Dados: Sorocaba Inteligente

Este notebook implementa uma pipeline de alta performance para análise de grandes volumes de dados (12GB+).

**Estrutura da Pipeline:**
1. **Setup e Dependências**: Configuração de ambiente e bibliotecas.
2. **Engenharia de Dados**: Conversão CSV para Parquet via stream (Otimização de Memória).
3. **EDA Global**: Diagnóstico completo de integridade e estatísticas sem carregar tudo na RAM.

## 1. Setup e Dependências

### 1.1 Conexão com Storage
Montagem do Google Drive para persistência de dados e acesso ao dataset original.

Para garantir que o Google Colab possa acessar os arquivos armazenados no seu Google Drive, é essencial montar o Drive. Esta etapa estabelece a conexão, permitindo que você referencie seus arquivos usando caminhos como `/content/drive/MyDrive/`.

In [1]:
# Montar o Google Drive para acessar os arquivos

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Engenharia de Dados: Conversão para Parquet
Crucial para permitir análises em segundos sem estourar os 12GB de RAM.

Esta seção é dedicada à preparação do ambiente Python, garantindo que todas as bibliotecas necessárias para as diferentes fases do projeto estejam disponíveis. Incluiremos bibliotecas para:

*   **Manipulação e Análise de Dados**: `pandas`, `numpy`
*   **Visualização de Dados (Descritiva e Diagnóstica)**: `matplotlib.pyplot`, `seaborn`, `plotly.express`, `plotly.graph_objects`
*   **Pré-processamento e Modelagem (Preditiva e PCA)**: `sklearn.model_selection`, `sklearn.preprocessing`, `sklearn.compose`, `sklearn.pipeline`, `sklearn.metrics`, `sklearn.decomposition.PCA`, e um modelo base como `RandomForestClassifier` para análise preditiva.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
import pyarrow as pa
import os
import gc
from google.colab import drive

# Montagem do Drive
drive.mount('/content/drive')

# Configurações de Caminho
BASE_PATH = '/content/drive/MyDrive/datasets/projeto-cidade-inteligente/'
CSV_FILE = os.path.join(BASE_PATH, 'DADOS_SOROCABA_JANEIRO_A_ABRIL_2026.csv')
PARQUET_FILE = os.path.join(BASE_PATH, 'DADOS_SOROCABA_JANEIRO_A_ABRIL_2026.parquet')

print("Ambiente configurado.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ambiente configurado com sucesso.


## 2.1 Finalização da Conversão de Dados

Nesta etapa crucial, os dados serão carregados a partir do arquivo CSV armazenado no seu Google Drive para um DataFrame pandas. É fundamental que o caminho para o arquivo (`data_path`) esteja correto e que o Google Drive tenha sido montado com sucesso na etapa anterior.

**Caminho do arquivo**: O caminho `'/content/drive/MyDrive/datasets/projeto-cidade-inteligente/DADOS_SOROCABA_JANEIRO_A_ABRIL_2026.csv'` será utilizado conforme definido.

**Considerações sobre Encoding e Separador**: A célula tentará carregar o arquivo usando `encoding='latin1'` e `sep=';'`, que foram identificados como potenciais configurações corretas em tentativas anteriores para lidar com possíveis erros de decodificação. Se você ainda encontrar problemas, pode ser necessário ajustar esses parâmetros (por exemplo, `encoding='utf-8'`, `sep=','`) ou verificar o formato do seu arquivo CSV.

### Resolvendo o Problema de Memória com Datasets Grandes

Como o dataset excede a memória RAM disponível no Colab, precisamos adotar uma estratégia de carregamento e processamento em partes. Vamos carregar o arquivo em 'chunks' (pedaços) para evitar esgotar a memória.

Primeiro, vamos tentar carregar um pedaço pequeno do arquivo para inspecionar os tipos de dados e otimizar o uso de memória. Isso nos permitirá definir os tipos de dados corretos antes de processar o arquivo completo em chunks, economizando ainda mais memória.

In [3]:
def robust_csv_to_parquet(csv_path, parquet_path, chunk_size=250000):
    if os.path.exists(parquet_path):
        print(f"[INFO] Parquet já existe: {parquet_path}")
        return True

    forced_dtypes = {
        'NSerie': 'str', 'Endereco': 'str', 'Sentido': 'str',
        'Datatrafego': 'str', 'Latitude': 'str', 'Longitude': 'str',
        'Placa': 'str', 'Velocidade 1': 'float64', 'Velocidade Regul': 'float64'
    }

    print(f"[START] Convertendo 12GB para Parquet...")
    try:
        reader = pd.read_csv(csv_path, encoding='latin1', sep=';', chunksize=chunk_size, dtype=forced_dtypes, low_memory=False)
        writer = None
        total_rows = 0

        for i, chunk in enumerate(reader):
            table = pa.Table.from_pandas(chunk)
            if writer is None:
                writer = pq.ParquetWriter(parquet_path, table.schema, compression='snappy')
            writer.write_table(table)
            total_rows += len(chunk)
            del chunk; del table; gc.collect()
            if i % 20 == 0: print(f"[PROGRESS] {total_rows:,} linhas processadas...")

        if writer: writer.close()
        print(f"[SUCCESS] {total_rows:,} registros convertidos.")
        return True
    except Exception as e:
        print(f"[CRITICAL] Erro: {e}")
        return False

robust_csv_to_parquet(CSV_FILE, PARQUET_FILE)

[INFO] Removendo arquivo Parquet antigo para nova conversão total...
[START] Iniciando conversão TOTAL do arquivo de 12GB...
[PROGRESS] 250,000 linhas processadas e gravadas no disco...
[PROGRESS] 2,750,000 linhas processadas e gravadas no disco...
[PROGRESS] 5,250,000 linhas processadas e gravadas no disco...
[PROGRESS] 7,750,000 linhas processadas e gravadas no disco...
[PROGRESS] 10,250,000 linhas processadas e gravadas no disco...
[PROGRESS] 12,750,000 linhas processadas e gravadas no disco...
[PROGRESS] 15,250,000 linhas processadas e gravadas no disco...
[PROGRESS] 17,750,000 linhas processadas e gravadas no disco...
[PROGRESS] 20,250,000 linhas processadas e gravadas no disco...
[PROGRESS] 22,750,000 linhas processadas e gravadas no disco...
[PROGRESS] 25,250,000 linhas processadas e gravadas no disco...
[PROGRESS] 27,750,000 linhas processadas e gravadas no disco...
[PROGRESS] 30,250,000 linhas processadas e gravadas no disco...
[PROGRESS] 32,750,000 linhas processadas e gravad

In [2]:
import os

# Garantindo que as variáveis de caminho existam nesta célula
BASE_PATH = '/content/drive/MyDrive/datasets/projeto-cidade-inteligente/'
PARQUET_FILE = os.path.join(BASE_PATH, 'DADOS_SOROCABA_JANEIRO_A_ABRIL_2026.parquet')

if os.path.exists(PARQUET_FILE):
    size_gb = os.path.getsize(PARQUET_FILE) / (1024**3)
    print(f"--- VERIFICAÇÃO DE ARQUIVO ---")
    print(f"Arquivo Parquet verificado: {size_gb:.2f} GB")
else:
    print(f"[ERRO] Arquivo não encontrado. Verifique a Seção 2.")

NameError: name 'PARQUET_FILE' is not defined

## 3. EDA Global: Diagnóstico de 100% dos Dados
Processamento em stream para extrair informações sem carregar o arquivo na RAM.

### 3.1 Integridade e Diagnóstico Estatístico
Nesta etapa, validamos a qualidade dos dados carregados e observamos as métricas descritivas principais.

## 4. Pré-processamento e Limpeza de Dados

Tratamento de valores nulos, normalização e engenharia de atributos para preparar o dataset para os modelos.

Começaremos com uma visão geral do DataFrame, incluindo seus tipos de dados, contagem de valores não nulos e uso de memória. Isso nos dará uma primeira ideia da integridade e da estrutura dos dados.

In [ ]:
import pyarrow.dataset as ds
import pyarrow.compute as pc

# Usamos a API de Dataset do PyArrow para analisar os 12GB sem carregar no Pandas
dataset = ds.dataset(PARQUET_FILE, format="parquet")

print("--- ANÁLISE DO DATASET COMPLETO (12GB) ---")
print(f"Total de Linhas detectadas: {dataset.count_rows():,}")

# Exemplo: Calcular média de velocidade em TODO o dataset sem estourar a RAM
print("Calculando média global de 'Velocidade 1'...")
# O PyArrow processa isso em blocos internos eficientes
avg_vel = pc.mean(dataset.to_table(columns=['Velocidade 1'])['Velocidade 1']).as_py()
print(f"Velocidade Média Global: {avg_vel:.2f} km/h")

# Se precisar de Pandas para EDA, use uma amostra MAIOR agora que o Parquet está pronto
df_eda = dataset.to_table().slice(0, 1000000).to_pandas() # 1 milhão de linhas para análise
print("Amostra de 1 milhão de linhas carregada para df_eda.")
display(df_eda.head())

--- ANÁLISE DO DATASET COMPLETO (12GB) ---
Total de Linhas detectadas: 76,662,221
Calculando média global de 'Velocidade 1'...
Velocidade Média Global: 42.07 km/h


### 3.1.1 Estatísticas Globais (Arquivo Inteiro)
Como o dataset é massivo, extraímos o número total de registros diretamente dos metadados do arquivo Parquet, sem carregar as linhas na RAM.

In [1]:
parquet_file = pq.ParquetFile(PARQUET_FILE)
metadata = parquet_file.metadata
schema = parquet_file.schema_arrow

print(f"--- ESTATÍSTICAS GLOBAIS ---")
print(f"Total de Linhas: {metadata.num_rows:,}")
print(f"Tamanho Parquet: {os.path.getsize(PARQUET_FILE) / (1024**3):.2f} GB")

print("\n--- ESTRUTURA E VARIÁVEIS ---")
for field in schema:
    print(f"Coluna: {field.name:20} | Tipo: {field.type}")

# Diagnóstico de Nulos e Estatísticas
total_nulls = {col: 0 for col in schema.names}
numeric_cols = ['Velocidade 1', 'Velocidade Regul']
stats = {col: {'min': np.inf, 'max': -np.inf, 'sum': 0, 'count': 0} for col in numeric_cols}

print("\n[INFO] Escaneando nulos e estatísticas (Streaming)...")
for batch in parquet_file.iter_batches(batch_size=1000000):
    df_batch = batch.to_pandas()
    for col in df_batch.columns:
        total_nulls[col] += df_batch[col].isna().sum()
        if col in numeric_cols:
            valid = df_batch[col].dropna()
            if not valid.empty:
                stats[col]['min'] = min(stats[col]['min'], valid.min())
                stats[col]['max'] = max(stats[col]['max'], valid.max())
                stats[col]['sum'] += valid.sum()
                stats[col]['count'] += valid.count()

print("\n--- CONTAGEM DE NULOS ---")
for col, count in total_nulls.items():
    print(f"{col:20}: {count:,} nulos ({(count/metadata.num_rows)*100:.2f}%)")

print("\n--- RESUMO NUMÉRICO ---")
for col in numeric_cols:
    mean = stats[col]['sum'] / stats[col]['count'] if stats[col]['count'] > 0 else 0
    print(f"{col}: Min={stats[col]['min']}, Max={stats[col]['max']}, Média={mean:.2f}")

NameError: name 'PARQUET_FILE' is not defined

### 3.1.2 Análise de Valores Nulos e Únicos (Processamento Otimizado)
Para saber quantos nulos existem no total de 12GB, percorremos o arquivo em lotes.

In [ ]:
import numpy as np

# Para análise completa sem estourar a RAM, processamos em batches (lotes)
total_nulls = {}
cols_numeric = ['Velocidade 1', 'Velocidade Regul']
stats = {col: {'sum': 0, 'count': 0, 'min': np.inf, 'max': -np.inf} for col in cols_numeric}

print("[INFO] Escaneando 100% do dataset para diagnóstico de nulos e estatísticas...")

# iter_batches lê o arquivo do disco em pedaços pequenos
for batch in parquet_file.iter_batches(batch_size=1000000):
    df_batch = batch.to_pandas()

    # Contagem de nulos total
    for col in df_batch.columns:
        total_nulls[col] = total_nulls.get(col, 0) + df_batch[col].isna().sum()

    # Estatísticas básicas para colunas numéricas
    for col in cols_numeric:
        stats[col]['sum'] += df_batch[col].sum()
        stats[col]['count'] += df_batch[col].count()
        stats[col]['min'] = min(stats[col]['min'], df_batch[col].min())
        stats[col]['max'] = max(stats[col]['max'], df_batch[col].max())

print("\n--- DIAGNÓSTICO DE NULOS (BASE INTEIRA) ---")
for col, count in total_nulls.items():
    print(f"{col:20}: {count:,} nulos")

print("\n--- ESTATÍSTICAS DESCRITIVAS GLOBAIS ---")
for col in cols_numeric:
    mean = stats[col]['sum'] / stats[col]['count'] if stats[col]['count'] > 0 else 0
    print(f"{col}: Min={stats[col]['min']}, Max={stats[col]['max']}, Média={mean:.2f}")

---

Saber a quantidade de linhas e colunas é fundamental para entender o tamanho do nosso conjunto de dados.

In [ ]:
# Exibir as dimensões do DataFrame (número de linhas e colunas)
print("\n### Dimensões do DataFrame (df.shape):")
print("------------------------------------")
print(f"O DataFrame possui {df.shape[0]} linhas e {df.shape[1]} colunas.")

---

A identificação de valores nulos é crucial para as etapas de limpeza e pré-processamento. Esta análise nos mostrará quais colunas possuem dados faltantes e em que quantidade.

In [ ]:
# Contagem de valores nulos por coluna
print("\n### Contagem de Valores Nulos por Coluna (df.isnull().sum()):")
print("------------------------------------------------------------")
null_counts = df.isnull().sum()
null_percentages = (df.isnull().sum() / len(df)) * 100
null_df = pd.DataFrame({
    'Contagem Nulos': null_counts,
    'Percentual Nulos (%)': null_percentages
})

# Filtrar apenas colunas com valores nulos para uma visualização mais limpa
null_df = null_df[null_df['Contagem Nulos'] > 0].sort_values(by='Contagem Nulos', ascending=False)

if not null_df.empty:
    print("As seguintes colunas possuem valores nulos:")
    display(null_df)
else:
    print("Não há valores nulos em nenhuma coluna do DataFrame.")

---

Esta análise fornece um resumo estatístico das colunas numéricas, incluindo média, desvio padrão, valores mínimo e máximo, e quartis. Essas métricas são essenciais para entender a distribuição e a variabilidade dos dados numéricos.

In [ ]:
# Exibir estatísticas descritivas para colunas numéricas
print("\n### Estatísticas Descritivas para Colunas Numéricas (df.describe()):")
print("-------------------------------------------------------------------")
display(df.describe())

---

Para colunas não numéricas (categóricas ou de texto), é importante verificar a quantidade de valores únicos (cardinalidade) e a distribuição desses valores. Isso ajuda a identificar colunas com muitos valores únicos (alta cardinalidade), que podem precisar de tratamento especial, ou colunas com poucos valores únicos (categóricas nominais/ordinais).

In [ ]:
# Análise de colunas categóricas e de alta cardinalidade
print("\n### Análise de Colunas Categóricas e de Alta Cardinalidade:")
print("----------------------------------------------------------")

# Identificar colunas não numéricas
categorical_cols = df.select_dtypes(include='object').columns

if not categorical_cols.empty:
    for col in categorical_cols:
        print(f"\n----- Coluna: '{col}' -----")
        unique_values = df[col].nunique()
        total_rows = len(df)
        print(f"  Número de valores únicos: {unique_values}")
        print(f"  Percentual de valores únicos: {(unique_values / total_rows * 100):.2f}%")

        # Exibir os 10 valores mais frequentes para colunas com até 50 valores únicos (baixa/média cardinalidade)
        if unique_values <= 50:
            print("  Valores mais frequentes (Top 10):")
            display(df[col].value_counts().head(10))
        else:
            print("  Esta coluna possui alta cardinalidade (> 50 valores únicos), exibindo apenas os 10 mais frequentes:")
            display(df[col].value_counts().head(10))
else:
    print("Não há colunas de tipo 'object' (categóricas) no DataFrame.")

---

Se o dataset contém colunas de data/hora, é importante convertê-las para o tipo `datetime` para permitir análises temporais, como verificar o período coberto pelos dados.

In [ ]:
# Tentativa de identificar e converter colunas que podem ser de data/hora
print("\n### Análise de Colunas de Data/Hora:")
print("----------------------------------")

date_cols_identified = []
for col in df.columns:
    # Tentativa heurística: Se a coluna contém 'data' ou 'dt' e não é numérica, pode ser uma data.
    # Melhorar isso com base no conhecimento do seu dataset.
    if ('data' in col.lower() or 'dt' in col.lower()) and df[col].dtype == 'object':
        try:
            # Tenta converter para datetime. errors='coerce' transformará valores inválidos em NaT (Not a Time)
            df[col] = pd.to_datetime(df[col], errors='coerce')
            if df[col].isnull().sum() / len(df) < 0.5: # Se menos de 50% dos valores virarem NaT, consideramos a conversão bem-sucedida
                date_cols_identified.append(col)
                print(f"Coluna '{col}' convertida para datetime com sucesso.")
                print(f"  Período da coluna '{col}': de {df[col].min()} a {df[col].max()}")
            else:
                print(f"Coluna '{col}' parece ser de data/hora, mas a conversão resultou em muitos valores inválidos. Mantendo como 'object'.")
                df[col] = df[col].astype(str) # Reverta se a conversão resultou em muitos nulos
        except Exception as e:
            print(f"Não foi possível converter a coluna '{col}' para datetime: {e}")

if not date_cols_identified:
    print("Nenhuma coluna de data/hora foi identificada ou convertida com sucesso.\nVocê pode precisar identificar manualmente e aplicar `pd.to_datetime()` com o formato correto, ex: `pd.to_datetime(df['sua_coluna_data'], format='%Y-%m-%d')`.")

---

Com base nas informações coletadas até agora, podemos começar a formular perguntas de negócio e planejar as próximas etapas da EDA, como visualizações mais aprofundadas e análise de correlações.

**Exemplos de Perguntas de Negócio que podem surgir:**
*   Qual a distribuição dos eventos ao longo do tempo?
*   Existem padrões geográficos nos dados?
*   Quais são as categorias mais comuns ou importantes?
*   Há alguma anomalia nos dados que mereça investigação?
*   Quais variáveis podem ser preditores de um evento específico?

**Próximos Passos Sugeridos na EDA:**
1.  **Visualização das Distribuições**: Utilizar histogramas e boxplots para colunas numéricas, e gráficos de barras para colunas categóricas.
2.  **Relação entre Variáveis**: Investigar correlações entre variáveis numéricas e a relação entre variáveis categóricas e numéricas.
3.  **Análise Temporal**: Se houver colunas de data/hora, analisar tendências, sazonalidade e padrões ao longo do tempo.
4.  **Análise Geoespacial**: Se houver dados de localização, mapear eventos para identificar clusters ou áreas de interesse.

## 5. Mineração de Dados

Execução de algoritmos de Machine Learning (Clusterização/Classificação) para extração de conhecimento.

Após a EDA, a fase de Pré-processamento e Limpeza dos Dados se torna crucial. Aqui, transformaremos os dados brutos em um formato adequado para a modelagem. As etapas comuns incluem:

*   **Tratamento de Valores Ausentes**: Decidir se removeremos, preencheremos (com média, mediana, moda) ou interpolaremos os valores nulos.
*   **Correção de Tipos de Dados**: Garantir que todas as colunas estejam no formato correto.
*   **Tratamento de Outliers**: Identificar e lidar com valores extremos que podem enviesar a análise.
*   **Codificação de Variáveis Categóricas**: Transformar variáveis de texto em representações numéricas (e.g., One-Hot Encoding, Label Encoding).
*   **Normalização/Escalonamento**: Ajustar a escala de variáveis numéricas para que nenhuma delas domine o processo de modelagem, especialmente para algoritmos sensíveis à escala.

## 6. Avaliação e Resultados

Análise de métricas de performance (Acurácia, F1-Score, RMSE) e validação dos modelos.

Nesta fase, aplicaremos algoritmos de Machine Learning para construir modelos preditivos ou descritivos, dependendo do objetivo do projeto. As etapas típicas incluem:

*   **Seleção do Modelo**: Escolher o algoritmo mais adequado (classificação, regressão, clusterização, etc.).
*   **Divisão dos Dados**: Separar o dataset em conjuntos de treino, validação e teste.
*   **Treinamento do Modelo**: Ajustar os parâmetros do modelo aos dados de treino.
*   **Avaliação do Modelo**: Verificar o desempenho usando métricas apropriadas (acurácia, precisão, recall, F1-score, RMSE, etc.) nos dados de teste.
*   **Otimização do Modelo**: Ajustar hiperparâmetros para melhorar o desempenho (e.g., Grid Search, Random Search).
*   **PCA (Análise de Componentes Principais)**: Utilizar PCA para redução de dimensionalidade, o que pode ajudar na visualização e na melhoria do desempenho de alguns modelos.

## 7. Conclusão e Próximos Passos

Insights finais e recomendações estratégicas baseadas na análise de dados.

A etapa final do projeto envolve a sumarização dos achados mais importantes, a avaliação crítica dos modelos desenvolvidos e a proposição de direções para trabalhos futuros. Esta é a oportunidade de consolidar todo o conhecimento e apresentar recomendações claras com base na análise de dados.

---

In [ ]:
import os

csv_size = os.path.getsize(CSV_FILE) / (1024**3)
pq_size = os.path.getsize(PARQUET_FILE) / (1024**3)

print("=== RELATÓRIO DE CONVERSÃO ===")
print(f"[OK] O arquivo CSV foi convertido com sucesso para Parquet.")
print(f"Tamanho Original (CSV): {csv_size:.2f} GB")
print(f"Tamanho Otimizado (Parquet): {pq_size:.2f} GB")
print(f"Localização: {PARQUET_FILE}")
print("==============================")

In [ ]:
# BLOCO REMOVIDO PARA EVITAR ESTOURO DE MEMÓRIA

In [ ]:
# BLOCO REMOVIDO PARA EVITAR ESTOURO DE MEMÓRIA

---

## 1. Setup Inicial

---

In [ ]:
# Montar o Google Drive para acessar os arquivos
from google.colab import drive
drive.mount('/content/drive')

---

### Instalação de Bibliotecas Necessárias

In [ ]:
# Instalar bibliotecas, se necessário (descomente para instalar)
# !pip install pandas numpy matplotlib seaborn plotly scikit-learn

---

### Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Todas as bibliotecas foram importadas com sucesso!")

---

### 2.2 Verificação do Dataset Otimizado

O carregamento direto via `pd.read_csv` foi desativado para evitar estouro de memória. O sistema agora utiliza o arquivo Parquet gerado na seção 2.

Nesta etapa, carregaremos os dados do arquivo CSV para um DataFrame pandas. É crucial que o caminho para o arquivo (`data_path`) esteja correto e que o Google Drive esteja montado.

**Observação Importante:** A célula abaixo tenta carregar o arquivo usando `encoding='latin1'` e `sep=';'`. Se você encontrar erros de codificação ou de parse, pode ser necessário ajustar esses parâmetros (ex: `encoding='utf-8'`, `sep=','`) ou verificar se o arquivo CSV está bem formatado.

**Certifique-se de executar esta célula para que o DataFrame `df` esteja disponível para as análises subsequentes.**

In [ ]:
# CÉLULA DESATIVADA PARA PROTEÇÃO DA RAM
# O carregamento via CSV direto consome 12GB+ e crasha a VM.
# Utilize o DataFrame 'df' carregado via Parquet na Seção 2.

---

### 3.2 Detalhamento de Atributos
Investigação profunda de variáveis específicas e suas correlações.

Nesta seção, você explorará as características do seu conjunto de dados, buscando padrões, anomalias e insights iniciais.

A célula a seguir apresenta um resumo do DataFrame, incluindo:

*   **`df.info()`**: Detalhes sobre os tipos de dados de cada coluna e a presença de valores não nulos.
*   **`df.shape`**: O número de linhas e colunas.
*   **`df.isnull().sum()`**: A contagem de valores ausentes por coluna.
*   **`df.describe()`**: Estatísticas descritivas para colunas numéricas (média, desvio padrão, quartis, etc.).

Execute a célula abaixo para obter essas informações e iniciar sua compreensão dos dados.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações de estilo
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (15, 6)

print("--- EDA: DISTRIBUIÇÃO DE VELOCIDADES (Amostra 1M) ---")

fig, ax = plt.subplots(1, 2, figsize=(18, 6))

# 1. Histograma de Velocidade
sns.histplot(df_eda['Velocidade 1'], bins=50, kde=True, ax=ax[0], color='skyblue')
ax[0].set_title('Distribuição de Velocidade Real')
ax[0].set_xlabel('km/h')

# 2. Boxplot para identificar Outliers
sns.boxplot(x=df_eda['Velocidade 1'], ax=ax[1], color='salmon')
ax[1].set_title('Análise de Outliers de Velocidade')

plt.tight_layout()
plt.show()

# 3. Análise de Sentido de Tráfego
plt.figure(figsize=(10, 5))
sns.countplot(data=df_eda, x='Sentido', palette='viridis')
plt.title('Volumetria por Sentido de Tráfego')
plt.xticks(rotation=45)
plt.show()

---

## 4. Pré-processamento e Limpeza dos Dados

A fase de **Pré-processamento e Limpeza dos Dados** é fundamental para garantir a qualidade dos dados. Aqui, você lidará com:

*   **Valores Ausentes**: Estratégias como remoção de linhas/colunas, preenchimento com média, mediana, moda ou interpolação.
*   **Tipos de Dados**: Conversão de colunas para os tipos corretos (ex: data para datetime, categóricas para tipo 'category').
*   **Outliers**: Identificação e tratamento de valores extremos que podem distorcer a análise.
*   **Codificação de Variáveis Categóricas**: Transformação de variáveis textuais em formatos numéricos que modelos de ML possam entender (ex: One-Hot Encoding).
*   **Normalização/Escalonamento**: Ajuste da escala de variáveis numéricas para evitar que características com grandes magnitudes dominem o treinamento do modelo.

Aqui você tratará valores ausentes, converterá tipos de dados, lidará com outliers, etc.

---

## 5. Mineração de Dados

Na etapa de **Mineração de Dados**, o foco é aplicar algoritmos de Machine Learning para construir modelos preditivos ou descritivos. Esta fase envolve:

*   **Seleção do Modelo**: Escolha do algoritmo mais adequado ao problema (classificação, regressão, clusterização, etc.).
*   **Divisão dos Dados**: Separação do dataset em conjuntos de treino, validação e teste.
*   **Treinamento do Modelo**: Ajuste dos parâmetros do modelo aos dados de treino.
*   **Avaliação do Modelo**: Verificação do desempenho do modelo usando métricas apropriadas (acurácia, precisão, recall, F1-score, RMSE, etc.) nos dados de teste.
*   **Otimização do Modelo**: Ajuste de hiperparâmetros para melhorar o desempenho (Grid Search, Random Search).

Nesta seção, você construirá, treinará e avaliará seus modelos de Machine Learning para a análise preditiva.

---

## 6. Conclusão e Próximos Passos

A seção de **Conclusão e Próximos Passos** sumariza os principais achados do projeto, avalia a eficácia dos modelos desenvolvidos e propõe direções para trabalhos futuros. Esta é a oportunidade de consolidar o conhecimento obtido e apresentar as recomendações.

Resumo das descobertas, avaliação do modelo e sugestões para trabalhos futuros.

---